In [1]:
import torch

In [2]:
# requires_grad=True tells PyTorch: "Track the operations performed on this tensor and build a computational graph, because later I want to get the derivative (gradient) with respect to this tensor via .backward()."

# In other words, this is required in order to use .backward(), but let's state it a bit more precisely:

a = torch.tensor([2.0, 6.0, 7.0], requires_grad=True)
b = (a ** 2).sum()
c = b * 3
d = c / 2
print(b) 
print(c)
print(d) 

tensor(89., grad_fn=<SumBackward0>)
tensor(267., grad_fn=<MulBackward0>)
tensor(133.5000, grad_fn=<DivBackward0>)


In [3]:
c.backward()
print(b)
print(c)
print(d)     
print(a.grad)     

tensor(89., grad_fn=<SumBackward0>)
tensor(267., grad_fn=<MulBackward0>)
tensor(133.5000, grad_fn=<DivBackward0>)
tensor([12., 36., 42.])


In [4]:
import pandas as pd
df=pd.read_csv("avg-household-size.csv")    

In [5]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

TARGET_COL='avghouseholdsize'
BATCH_SIZE = 64
EPOCHS = 50
ALPHA = 0.001

x = df.drop(columns=[TARGET_COL]) # Features

y = df[TARGET_COL] # Target

x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=42
)

# Encoding

In [6]:
def encode_train(df):
    df = df.copy()
    encoders = {}
    for col in df.columns:
        if not pd.api.types.is_numeric_dtype(df[col]):
            df[col] = df[col].astype(str)
            encoder = LabelEncoder()
            df[col] = encoder.fit_transform(df[col])
            encoders[col] = encoder
    return df, encoders

def encode_test(df, encoders):
    df = df.copy()
    for col, encoder in encoders.items():
        df[col] = df[col].astype(str)
        known = set(encoder.classes_)
        df[col] = df[col].apply(lambda x: encoder.transform([x])[0] if x in known else -1)
    return df

encoded_xtrain, encoders = encode_train(x_train)   # train
encoded_xtest = encode_test(x_test, encoders)       # test



# Pandas to Numpy

In [7]:
x_train = encoded_xtrain.to_numpy()
x_test = encoded_xtest.to_numpy()
y_train_arr = y_train.to_numpy().reshape(-1, 1)
y_test_arr = y_test.to_numpy().reshape(-1, 1)

# Scaling

In [8]:
# Scaling the features and target variable
from sklearn.preprocessing import StandardScaler

x_scaler = StandardScaler()
scaled_xtrain = x_scaler.fit_transform(encoded_xtrain)
scaled_xtest  = x_scaler.transform(encoded_xtest)

y_scaler = StandardScaler()
scaled_ytrain = y_scaler.fit_transform(y_train_arr)
scaled_ytest  = y_scaler.transform(y_test_arr)

In [9]:
scaled_xtrain

array([[-0.22257395, -0.29941883, -0.03832566],
       [ 0.82921827, -0.2808567 ,  0.5923668 ],
       [-1.21249604, -0.11379753,  0.56412684],
       ...,
       [-0.34631421,  0.20175868,  0.55067924],
       [-0.84127526,  0.22032081,  0.78601225],
       [ 1.38604944, -0.37366734,  0.43368511]], shape=(2576, 3))

In [10]:
scaled_xtest

array([[-1.45997656,  0.03469951, -1.73272332],
       [-1.15062591,  1.18555155, -1.73272332],
       [-0.77940513,  0.51731488, -1.73272332],
       ...,
       [-1.64558696, -0.24373244, -1.73272332],
       [-0.16070382, -0.16948392, -1.73272332],
       [ 2.49971179, -0.54072651, -1.73272332]], shape=(644, 3))

In [11]:
scaled_ytrain

array([[ 0.52633053],
       [ 0.33890104],
       [ 0.86370361],
       ...,
       [ 0.22644335],
       [-0.11092974],
       [-0.67321821]], shape=(2576, 1))

In [12]:
scaled_ytest

array([[-5.60760513e-01],
       [ 9.01189511e-01],
       [-1.08556309e+00],
       [-1.85901532e-01],
       [-4.10816921e-01],
       [-7.34438383e-02],
       [-6.73218207e-01],
       [ 8.26217715e-01],
       [ 5.26330531e-01],
       [-1.85901532e-01],
       [ 1.88957448e-01],
       [-3.73331023e-01],
       [ 3.01415142e-01],
       [ 6.38788225e-01],
       [ 3.76386938e-01],
       [ 2.26443346e-01],
       [-8.98133596e-01],
       [-5.98246411e-01],
       [ 1.23856259e+00],
       [-2.60873329e-01],
       [ 2.92542801e+00],
       [-2.98359227e-01],
       [ 7.13760021e-01],
       [ 2.63929244e-01],
       [ 1.13985652e-01],
       [-4.85788717e-01],
       [-1.27299258e+00],
       [ 5.26330531e-01],
       [-4.48302819e-01],
       [ 1.46347798e+00],
       [ 1.88957448e-01],
       [-1.10929736e-01],
       [-3.35845125e-01],
       [-4.10816921e-01],
       [ 7.64997539e-02],
       [-1.12304898e+00],
       [-1.10929736e-01],
       [ 1.51471550e-01],
       [ 6.3

# Numpy to tensor

In [13]:
x_train_tensor = torch.tensor(scaled_xtrain, dtype=torch.float32)
x_test_tensor = torch.tensor(scaled_xtest, dtype=torch.float32)

y_train_tensor = torch.tensor(scaled_ytrain, dtype=torch.float32)
y_test_tensor = torch.tensor(scaled_ytest, dtype=torch.float32)


In [14]:
x_train_tensor

tensor([[-0.2226, -0.2994, -0.0383],
        [ 0.8292, -0.2809,  0.5924],
        [-1.2125, -0.1138,  0.5641],
        ...,
        [-0.3463,  0.2018,  0.5507],
        [-0.8413,  0.2203,  0.7860],
        [ 1.3860, -0.3737,  0.4337]])

In [15]:
x_test_tensor

tensor([[-1.4600,  0.0347, -1.7327],
        [-1.1506,  1.1856, -1.7327],
        [-0.7794,  0.5173, -1.7327],
        ...,
        [-1.6456, -0.2437, -1.7327],
        [-0.1607, -0.1695, -1.7327],
        [ 2.4997, -0.5407, -1.7327]])

In [16]:
y_train_tensor

tensor([[ 0.5263],
        [ 0.3389],
        [ 0.8637],
        ...,
        [ 0.2264],
        [-0.1109],
        [-0.6732]])

In [17]:
y_test_tensor

tensor([[-5.6076e-01],
        [ 9.0119e-01],
        [-1.0856e+00],
        [-1.8590e-01],
        [-4.1082e-01],
        [-7.3444e-02],
        [-6.7322e-01],
        [ 8.2622e-01],
        [ 5.2633e-01],
        [-1.8590e-01],
        [ 1.8896e-01],
        [-3.7333e-01],
        [ 3.0142e-01],
        [ 6.3879e-01],
        [ 3.7639e-01],
        [ 2.2644e-01],
        [-8.9813e-01],
        [-5.9825e-01],
        [ 1.2386e+00],
        [-2.6087e-01],
        [ 2.9254e+00],
        [-2.9836e-01],
        [ 7.1376e-01],
        [ 2.6393e-01],
        [ 1.1399e-01],
        [-4.8579e-01],
        [-1.2730e+00],
        [ 5.2633e-01],
        [-4.4830e-01],
        [ 1.4635e+00],
        [ 1.8896e-01],
        [-1.1093e-01],
        [-3.3585e-01],
        [-4.1082e-01],
        [ 7.6500e-02],
        [-1.1230e+00],
        [-1.1093e-01],
        [ 1.5147e-01],
        [ 6.3879e-01],
        [ 5.2633e-01],
        [-3.3585e-01],
        [-1.4604e+00],
        [ 6.7627e-01],
        [-7

# Tensor Dataset

In [18]:
# Create a dataset from the tensors. Features and target variable are combined into a single dataset, which can be used for training a model. The TensorDataset class is used to create a dataset from the feature and target tensors. This dataset can then be used to create a DataLoader for batching and shuffling during training.
train_ds = torch.utils.data.TensorDataset(x_train_tensor, y_train_tensor) 

In [19]:
train_ds[10]

(tensor([1.0148, 2.0023, 0.2306]), tensor([0.6013]))

# Data Loader

In [20]:
# Create a data loader for the training dataset. to train the model, we need to feed the data in batches. The DataLoader class is used to create a data loader from the dataset. The batch size and shuffle parameters are specified to control the batching and shuffling of the data during training.
train_loader = torch.utils.data.DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)

# Model

In [21]:
# Set the random seed for reproducibility. 0 means that the random number generator will produce the same sequence of random numbers every time the code is run. This is important for reproducibility, as it allows us to obtain the same results when running the code multiple times. By setting the random seed to 0, we ensure that the random initialization of model parameters and any other random operations in the code will be consistent across different runs.

torch.manual_seed(0) # random seed for reproducibility

# to determine the number of input features for the model, we can use the shape of the training tensor. The shape of a tensor is a tuple that represents the dimensions of the tensor. In this case, we are interested in the second dimension, which represents the number of features in the training data. We can access this dimension using the shape attribute of the tensor and indexing it with [1]. This will give us the number of input features that we need to specify when defining our model architecture.

input_dim = x_train_tensor.shape[1] # features

In [22]:
input_dim

3

In [23]:
from torch import nn

model = nn.Sequential(
    nn.Linear(input_dim, 64),   #Intput
    nn.ReLU(),###Hidden nn.sigmoid()  nn.tanh()
    nn.Linear(64, 1)   #Output
)

# nn.Linear(input_features, output_features)
# input_features: The number of input features (or neurons) in the layer. This should match the number of features in your input data.
# output_features: The number of output features (or neurons) in the layer. This determines the number of neurons in the layer and the dimensionality of the output from this layer.

# nn.Sigmoid()   # 0 and 1 values
# nn.Tanh()      # -1 and 1 values
# nn.ReLU()      # 0 and more than 0 values, if negative then 0, if positive then same value

In [24]:
model


Sequential(
  (0): Linear(in_features=3, out_features=64, bias=True)
  (1): ReLU()
  (2): Linear(in_features=64, out_features=1, bias=True)
)

In [25]:
model.parameters()

<generator object Module.parameters at 0x11aa226c0>

In [26]:
# Loss function
loss_fn = nn.MSELoss()   

In [27]:
loss_fn

MSELoss()

# Optimizer

In [28]:
from torch import optim

opt = optim.Adam(model.parameters(), lr=ALPHA)


for epoch in range(1, EPOCHS + 1):
    model.train()  # Set the model to training mode
    total_loss = 0.0  # Initialize total loss for the epoch

    for bx, by in train_loader:
        # bx means batch of features, by means batch of target values

        pred = model(bx) # Forward pass: Compute the model's predictions for the batch of features (bx).
        loss = loss_fn(pred, by) # Compute the loss between the predictions (pred) and the true target values (by) using the specified loss function (loss_fn).

        opt.zero_grad() # Zero the gradients of the model's parameters to prevent accumulation from previous iterations.
        loss.backward() # Backward pass: Compute the gradients of the loss with respect to the model's parameters using backpropagation.

        # prevent exploding gradients
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        
        opt.step() # Update the model's parameters based on the computed gradients and the optimization algorithm (opt).
        total_loss += loss.item() # Accumulate the loss for the epoch by adding the current batch's loss to the total loss.

    train_loss = total_loss / len(train_loader) # Calculate the average training loss for the epoch by dividing the total loss by the number of batches in the training data loader (train_loader).

    model.eval() # Set the model to evaluation mode

    # Compute test loss
    with torch.no_grad():
        test_loss_after = loss_fn(model(x_test_tensor), y_test_tensor).item()

    if epoch % 10 == 0:
        print(f"Epoch {epoch:3d} | Train Loss: {train_loss:.4f} | Test Loss: {test_loss_after:.4f}")

print("AFTER GD (trained) Test Loss (scaled y):", round(test_loss_after, 4))





    
   

Epoch  10 | Train Loss: 0.9102 | Test Loss: 0.9873
Epoch  20 | Train Loss: 0.8854 | Test Loss: 1.0287
Epoch  30 | Train Loss: 0.8803 | Test Loss: 1.0708
Epoch  40 | Train Loss: 0.8916 | Test Loss: 1.0728
Epoch  50 | Train Loss: 0.8560 | Test Loss: 1.0542
AFTER GD (trained) Test Loss (scaled y): 1.0542
